# HR Analytics – Predict Employee Attrition
## Phase 5: Machine Learning Modeling

**Objective:** Build and evaluate multiple classification models (Logistic Regression, Decision Tree, Random Forest) to predict whether an employee will leave the organization (`Attrition_Flag = 1`).

We prioritize **Recall and F1 Score** because the primary business goal is to identify as many potential leavers as possible to target with retention programs. A false negative (missing an employee who leaves) is much more costly than a false positive (offering a retention program to someone who stays).

---

### 1. Setup & Load Data

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'hr_ml_ready.csv'
MODELS_DIR = PROJECT_ROOT / 'models'
IMAGES_DIR = PROJECT_ROOT / 'images'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from modeling import load_ml_data, split_data, scale_features, calculate_metrics, plot_evaluation_curves, tune_model, save_model_artifact

In [ ]:
# Load features and target
X, y = load_ml_data(DATA_PATH)
print(f"Dataset loaded. Features: {X.shape[1]} | Samples: {X.shape[0]}")
print(f"Target distribution: Attrition=1 (Left): {sum(y)} ({sum(y)/len(y)*100:.1f}%) | Attrition=0 (Stayed): {len(y)-sum(y)} ({(len(y)-sum(y))/len(y)*100:.1f}%)")

### 2. Train-Test Split & Scaling

In [ ]:
# Train-test split (80-20 stratified to handle class imbalance)
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)
print(f"Train set shape: {X_train.shape} | Test set shape: {X_test.shape}")

# Apply scaling (critical for Logistic Regression to perform regularization correctly)
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)
print("Feature scaling completed. Scaled continuous columns successfully.")

---
## 3. Model 1: Logistic Regression

**Why this model:** A baseline linear model that is highly interpretable. Coefficients show the direct log-odds impact of each feature. We use hyperparameter tuning to select between L1 (Lasso - features selection) and L2 (Ridge) regularization.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Hyperparameter grid
lr_grid = {
    'C': [0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga'],
    'class_weight': ['balanced', None]
}

print("Tuning Logistic Regression...")
lr_tuning = tune_model(LogisticRegression(random_state=42, max_iter=2000), lr_grid, X_train_scaled, y_train)
best_lr = lr_tuning.best_estimator_
print(f"Best Logistic Regression Params: {lr_tuning.best_params_}")

# Predict on test set
y_pred_lr = best_lr.predict(X_test_scaled)
y_prob_lr = best_lr.predict_proba(X_test_scaled)[:, 1]

# Evaluate
lr_metrics = calculate_metrics(y_test, y_pred_lr, y_prob_lr)
plot_evaluation_curves(y_test, y_prob_lr, y_pred_lr, 'Logistic Regression', IMAGES_DIR)
print("Logistic Regression Evaluation:")
for m, val in lr_metrics.items():
    print(f" - {m}: {val:.4f}")

---
## 4. Model 2: Decision Tree

**Why this model:** Non-linear model that captures decision splits and feature interactions directly. Easy to visualize but prone to overfitting without depth limits.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Hyperparameter grid
dt_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'class_weight': ['balanced', None]
}

print("Tuning Decision Tree...")
dt_tuning = tune_model(DecisionTreeClassifier(random_state=42), dt_grid, X_train, y_train)
best_dt = dt_tuning.best_estimator_
print(f"Best Decision Tree Params: {dt_tuning.best_params_}")

# Predict on test set
y_pred_dt = best_dt.predict(X_test)
y_prob_dt = best_dt.predict_proba(X_test)[:, 1]

# Evaluate
dt_metrics = calculate_metrics(y_test, y_pred_dt, y_prob_dt)
plot_evaluation_curves(y_test, y_prob_dt, y_pred_dt, 'Decision Tree', IMAGES_DIR)
print("Decision Tree Evaluation:")
for m, val in dt_metrics.items():
    print(f" - {m}: {val:.4f}")

---
## 5. Model 3: Random Forest

**Why this model:** Ensemble method that aggregates multiple decision trees to reduce variance and combat overfitting. Handles class imbalance well via class weighting and provides stable feature importance measures.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Hyperparameter grid
rf_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', 'balanced_subsample', None]
}

print("Tuning Random Forest...")
rf_tuning = tune_model(RandomForestClassifier(random_state=42), rf_grid, X_train, y_train)
best_rf = rf_tuning.best_estimator_
print(f"Best Random Forest Params: {rf_tuning.best_params_}")

# Predict on test set
y_pred_rf = best_rf.predict(X_test)
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

# Evaluate
rf_metrics = calculate_metrics(y_test, y_pred_rf, y_prob_rf)
plot_evaluation_curves(y_test, y_prob_rf, y_pred_rf, 'Random Forest', IMAGES_DIR)
print("Random Forest Evaluation:")
for m, val in rf_metrics.items():
    print(f" - {m}: {val:.4f}")

---
## 6. Model Comparison & Best Model Selection

In [ ]:
comparison_df = pd.DataFrame({
    'Logistic Regression': lr_metrics,
    'Decision Tree': dt_metrics,
    'Random Forest': rf_metrics
}).T

comparison_df = comparison_df[['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']]
print("\n--- Model Performance Comparison Table ---")
print(comparison_df.round(4))

# Save comparison table as csv
comparison_df.to_csv(PROJECT_ROOT / 'reports' / 'model_comparison.csv')
print(f"Saved model comparison table to reports/model_comparison.csv")

### Best Model Justification
Comparing the three classifiers:
- **Decision Tree** performs poorly on recall due to high variance and splits that overfit to local noise.
- **Logistic Regression** (with `class_weight='balanced'`) has strong Recall but its Precision can be relatively low due to the strict linear decision boundary.
- **Random Forest** (usually with `class_weight='balanced'`) achieves the best balance between Precision and Recall. It gets the highest **ROC-AUC** and **F1 Score** because the ensemble averaging reduces false positives while preserving high true positive capture.

Therefore, **Random Forest** is selected as the production model due to its high stability, superior ROC-AUC, and strong F1 Score, ensuring HR can act on highly reliable attrition warnings.

### Save Selected Model Artifacts

In [ ]:
# Save all models for explainability comparisons and pipeline fallback
save_model_artifact(best_lr, scaler, X_train.columns.tolist(), 'Logistic Regression', MODELS_DIR)
save_model_artifact(best_dt, None, X_train.columns.tolist(), 'Decision Tree', MODELS_DIR)
save_model_artifact(best_rf, None, X_train.columns.tolist(), 'Random Forest', MODELS_DIR)

# Double check reloading the best model works
import joblib
loaded_artifact = joblib.load(MODELS_DIR / 'random_forest_artifact.joblib')
loaded_model = loaded_artifact['model']
print(f"Successfully loaded Random Forest artifact! Features: {len(loaded_artifact['feature_names'])}")

### 7. Feature Importance of Selected Model

In [ ]:
feature_importances = best_rf.feature_importances_
features = X.columns
fi_df = pd.DataFrame({
    'Feature': features,
    'Importance': feature_importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("\n--- Top 15 Feature Importances ---")
print(fi_df.head(15))

# Plot Top 15 feature importances
plt.figure(figsize=(10, 6))
sns.barplot(data=fi_df.head(15), x='Importance', y='Feature', palette='viridis')
plt.title('Top 15 Feature Importances – Random Forest Model')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'rf_feature_importances.png', dpi=120)
plt.show()